In [19]:
import pandas as pd
import numpy as np

In [1]:
import pandas as pd
import numpy as np
import fitz
from sentence_transformers import SentenceTransformer
import faiss

/Users/shrinivasdachawar/Downloads/My_docccc/zzti/strategies/rag_envsource/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
df_ann=pd.read_parquet("/Users/shrinivasdachawar/Downloads/My_docccc/zzti/strategies/data/Announcements/announcements.parquet")

In [21]:
df_ann1=df_ann[df_ann['date']>'2026-04-01 18:40:24']

In [22]:
df_ann1.tail()

,symbol,seq_id,subject,company_name,attachment_text,industry,has_xbrl,date_raw,announcement_datetime,sort_datetime,attachment_url,date,index_name
163494,ASIANTILES,106596857,shareholders meeting,Asian Granito India Limited,Asian Granito India Limited has informed the E...,Ceramics And Sanitaryware,True,22042026150746,22-Apr-2026 15:07:46,2026-04-22 15:07:46,https://nsearchives.nseindia.com/corporate/ASI...,2026-04-22 15:07:46,equities
163495,LYPSAGEMS,106596861,general updates,Lypsa Gems & Jewellery Limited,Lypsa Gems & Jewellery Limited has informed th...,None,True,22042026151241,22-Apr-2026 15:12:41,2026-04-22 15:12:41,https://nsearchives.nseindia.com/corporate/LYP...,2026-04-22 15:12:41,equities
163496,HNDFDS,106596862,record date,Hindustan Foods Limited,Hindustan Foods Limited has informed the Excha...,None,True,22042026151311,22-Apr-2026 15:13:11,2026-04-22 15:13:11,https://nsearchives.nseindia.com/corporate/913...,2026-04-22 15:13:11,equities
163497,SENCO,106596863,copy of newspaper publication,Senco Gold Limited,Senco Gold Limited has informed the Exchange a...,None,True,22042026151351,22-Apr-2026 15:13:51,2026-04-22 15:13:51,https://nsearchives.nseindia.com/corporate/Sen...,2026-04-22 15:13:51,equities
163498,AGI,106596864,general updates,AGI Greenpac Limited,AGI Greenpac Limited has informed the Exchange...,None,True,22042026151533,22-Apr-2026 15:15:33,2026-04-22 15:15:33,https://nsearchives.nseindia.com/corporate/HSI...,2026-04-22 15:15:33,equities


In [23]:
df_tr=pd.DataFrame(df_ann1[(df_ann1['subject']=='analysts/institutional investor meet/con. call updates')]['attachment_text'].value_counts()).reset_index()

In [24]:
print(df_tr.columns)


Index(['attachment_text', 'count'], dtype='str')


In [25]:
results=df_tr[df_tr['attachment_text'].str.lower().str.contains("link", na=False)]

In [26]:
# 1. Filter by subject and "link" in attachment_text
mask = (df_ann1['subject'] == 'analysts/institutional investor meet/con. call updates') & \
       ((df_ann1['attachment_text'].str.lower().str.contains("link", na=False)) | df_ann1['attachment_text'].isnull())

# 2. Apply mask, sort by date (descending), and take top 200
df_tr1 = df_ann1[mask].sort_values(by='date', ascending=False)


In [27]:
df_share=df_ann[df_ann['subject'].str.lower().str.contains("analysts/institutional investor meet/con. call updates")].sort_values('date', ascending=False).head(50)

In [29]:
df_bulk=pd.read_parquet("/Users/shrinivasdachawar/Downloads/My_docccc/zzti/strategies/data/deals_data/bulk_deals.parquet")

In [32]:
df_bulk.head()

,Date,Symbol,Security Name,Client Name,Buy / Sell,Quantity Traded,Trade Price / Wght. Avg. Price,Remarks,Buy/Sell,fetch_date
0,2025-09-29,AAATECH,AAA Technologies Limited,CRAFT EMERGING MARKET FUND PCC- CITADEL CAPITA...,BUY,400000,78.60,-,None,None
1,2025-09-29,AAATECH,AAA Technologies Limited,RUCHI ANJAY AGARWAL,SELL,400000,78.60,-,None,None
2,2025-09-29,AETHER,Aether Industries Limited,AMANSA HOLDINGS PRIVATE LIMITED,BUY,1282613,735.00,-,None,None
3,2025-09-29,AETHER,Aether Industries Limited,GOLDMAN SACHS FDS GOLDMAN SACHS INDIA EQ PORTF...,SELL,1151084,735.01,-,None,None
4,2025-09-29,ANZEN,Anzen Ind Ene Yld Plu Tru,GO DIGIT GENERAL INSURANCE LIMITED,BUY,2175000,115.00,-,None,None


In [9]:
df_orders=pd.read_parquet("/Users/shrinivasdachawar/Downloads/My_docccc/zzti/strategies/data/Announcements/orders_100.parquet")

In [47]:
df_orders.columns

Index(['symbol', 'subject', 'company_name', 'date_raw',
       'announcement_datetime', 'sort_datetime', 'attachment_url', 'date',
       'date_only', 'pdf_text'],
      dtype='str')

In [119]:
import re

def smart_chunk_text(text, chunk_size=400, overlap=100):
    sentences = re.split(r'(?<=[.!?]) +', text)
    
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks

In [120]:
all_chunks = []

for _, row in df_orders.iterrows():
    chunks = smart_chunk_text(row["pdf_text"])
    
    for chunk in chunks:
        all_chunks.append({
            "text": chunk,
            "symbol": row["symbol"],
            "date": row["date"],
            "source": "announcement"
        })

In [121]:
chunks_df = pd.DataFrame(all_chunks)

In [122]:
all_chunks

[{'text': '',
  'symbol': 'ADANIPOWER',
  'date': Timestamp('2026-04-02 19:44:15'),
  'source': 'announcement'},
 {'text': 'date: april 2, 2026\nbse limited national stock exchange of india limited\nfloor 25, p j towers, exchange plaza,\ndalal street, bandra kurla complex,\nmumbai – 400 001 bandra (e), mumbai – 400 051\nscrip code: 533096 scrip code: adanipower\ndear sir,\nsub: intimation under regulation 30 of the sebi (listing obligations and disclosures\nrequirements) regulations, 2015 – letter of award\nadani power limited has received a letter of award (“loa”) from maharashtra state\nelectricity distribution co.',
  'symbol': 'ADANIPOWER',
  'date': Timestamp('2026-04-02 19:44:15'),
  'source': 'announcement'},
 {'text': 'limited (“msedcl”) for supply of 2500 mw re rtc power\nfor a period of 25 years from scheduled commencement date of supply, upon apl\nbeing declared as the successful bidder in the e-ra results.\nthe details, as required under sebi (listing obligations and disclo

In [123]:

(chunks_df.tail())

,text,symbol,date,source
774,kodanda rami reddy\ncompany secretary & compli...,LOKESHMACH,2026-03-17 19:25:33,announcement
775,sebi/ho/cfd/cfd-pod-1/p/cir/2023/123 dated: ju...,LOKESHMACH,2026-03-17 19:25:33,announcement
776,whether order(s) / contract(s) have been domes...,LOKESHMACH,2026-03-17 19:25:33,announcement
777,"9,50,01,741/-\nthe order(s)/contract(s); (incl...",LOKESHMACH,2026-03-17 19:25:33,announcement
778,"if yes,\nwhether the same is done at ""arm's le...",LOKESHMACH,2026-03-17 19:25:33,announcement


In [124]:
chunks_df[chunks_df['symbol']=='POWERMECH']

,text,symbol,date,source
94,"date: april 1, 2026\nto to\nlisting department...",POWERMECH,2026-04-01 10:38:37,announcement
95,services\nnational stock exchange of india lim...,POWERMECH,2026-04-01 10:38:37,announcement
96,the details\nof the order as required under th...,POWERMECH,2026-04-01 10:38:37,announcement
97,no particulars details\na name of the entity a...,POWERMECH,2026-04-01 10:38:37,announcement
98,"if no\nyes, whether the same is done at “arm’s...",POWERMECH,2026-04-01 10:38:37,announcement


In [125]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6614.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [126]:
texts = chunks_df["text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches: 100%|██████████| 25/25 [00:05<00:00,  4.89it/s]


In [127]:
print(len(embeddings))
print(embeddings[0].shape)

779
(384,)


In [128]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [129]:
print(index.ntotal)

779


In [130]:
def retrieve(query, k=5):
    q_emb = model.encode([query])
    
    distances, indices = index.search(q_emb, k)
    
    results = chunks_df.iloc[indices[0]]
    
    return results

In [131]:
output=retrieve("order received by adani power")

In [132]:
output

,text,symbol,date,source
3,"g. highway, khodiyar, www.adanipower.com\nahme...",ADANIPOWER,2026-04-02 19:44:15,announcement
1,"date: april 2, 2026\nbse limited national stoc...",ADANIPOWER,2026-04-02 19:44:15,announcement
2,limited (“msedcl”) for supply of 2500 mw re rt...,ADANIPOWER,2026-04-02 19:44:15,announcement
7,"if yes, whether the\nsame is done at “arm’s le...",ADANIPOWER,2026-04-02 19:44:15,announcement
95,services\nnational stock exchange of india lim...,POWERMECH,2026-04-01 10:38:37,announcement


In [133]:
retrieve("last week order received")

,text,symbol,date,source
622,the orders pertain to the supply of power dist...,MARINE,2026-03-24 10:21:14,announcement
653,the orders pertain to the supply of power dist...,MARINE,2026-03-23 13:42:25,announcement
157,"date: 31/03/2026\nto, to,\nnational stock exch...",INTERARCH,2026-03-31 16:06:47,announcement
511,sebi/ho/cfd/cfd-pod-1/p/cir/2023/123 dated jul...,PACEDIGITK,2026-03-26 12:53:10,announcement
350,"bandra (east), mumbai – 400 051, india.\nscrip...",AVANTEL,2026-03-28 13:04:59,announcement


In [78]:
print(df_orders[df_orders['symbol']=='POWERMECH']['pdf_text'].iloc[0])

date: april 1, 2026
to to
listing department dept. of corp. services
national stock exchange of india limited bse limited
exchange plaza, c-1, block g, phiroze jeejeebhoy towers
bandra kurla complex, dalal street
bandra (e), mumbai – 400 051 mumbai- 400001
symbol/security id: powermech security code: 539302
dear sir/madam,
sub: receipt of order
*****
we are pleased to inform you that the company has secured an order from hindustan zinc limited. the details
of the order as required under the sebi master circular no. sebi/ho/cfd/pod2/cir/p/0155 dated november
11, 2024, are given below:
s. no particulars details
a name of the entity awarding order(s)/ hindustan zinc limited
contract(s) (cin: l27204rj1966plc001208)
b significant terms and conditions of scope of work: the company is responsible for
order(s)/ contracts(s) awarded in brief comprehensive operation and maintenance of the 91.5 mw
cpp and transmission line up to the mrss, acting as an
independent contractor on behalf of hindustan